In [ ]:
import sys
import os
import random
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from anndata import AnnData
from scipy import sparse
import scanpy.external as sce
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import scrublet as scr

In [ ]:
sys.path.append(os.path.abspath("cd8treg_sc/Code/"))
import plots

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)

In [ ]:
from scipy.stats import median_abs_deviation


def is_outlier(
    adata: AnnData, metric: str, nmads: int = 5, side: str = "both"
) -> pd.Series:
    """Identify outliers based on MAD (median absolute deviations) with directional filtering.

    Parameters:
        adata (AnnData): Anndata object containing the data.
        metric (str): Name of the metric to check for outliers.
        nmads (int): Number of MADs to consider as threshold (default is 5).
        side (str): Direction for outlier detection ('left', 'right', or 'both'). Default: 'both'

    Returns:
        pd.Series: Boolean series indicating outliers.
    """
    if side not in ["left", "right", "both"]:
        raise ValueError("Parameter 'side' must be 'left', 'right', or 'both'")

    M = adata.obs[metric]
    median = np.median(M)
    mad = median_abs_deviation(M)

    lower_bound = median - nmads * mad
    upper_bound = median + nmads * mad

    if side == "left":
        outlier = M < lower_bound
    elif side == "right":
        outlier = M > upper_bound
    else:  # 'both'
        outlier = (M < lower_bound) | (M > upper_bound)

    return outlier

### Lung

In [ ]:
samplesheet = pd.read_csv("./h_lung/lung_samplesheet.csv", sep=";")
samplesheet.head(5)

In [ ]:
adata_list = []

In [ ]:
lung_exclusion = [
    "GSM4058905",
    "GSM4058908",
    "GSM7103281",
    "GSM7103327",
    "GSM7135597",
    "GSM7135598",
]  # by CR metrics
lung_exclusion.append("GSM6509487")  # no data

In [ ]:
for index, row in tqdm(samplesheet.iterrows(), total=len(samplesheet)):
    if row["GSM_id"] in lung_exclusion:
        continue
    path = f"./h_lung/cellranger_out/{row['GSM_id']}/outs/filtered_feature_bc_matrix"
    adata = sc.read_10x_mtx(path, prefix="")
    for col in row.index:
        adata.obs[col] = row[col]
    adata_list.append(adata)

In [ ]:
filtered_adata_list = []

In [ ]:
for num, adata in enumerate(adata_list):
    adata.obs.index = [f"{index}-{num}" for index in adata.obs.index]
    print(f"{num} - {adata.obs.GSM_id.unique()[0]}:")

    print(f"Total number of cells: {adata.n_obs}")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=None, log1p=True
    )

    plots.plot_qc_distributions(adata)
    plt.show()
    sc.pp.filter_cells(adata, min_counts=500)

    adata.obs["outlier"] = is_outlier(adata, "log1p_total_counts", 5) | is_outlier(
        adata, "n_genes_by_counts", 6
    )
    print(
        f"outlier counts: \n{adata.obs.outlier.value_counts().to_string(name=False, dtype=False)}\n"
    )

    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 4, "right") | (
        adata.obs["pct_counts_mt"] > 20
    )
    print(
        f"mt outlier counts: \n{adata.obs.mt_outlier.value_counts().to_string(name=False, dtype=False)}"
    )

    # filtering out:
    adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

    sc.pp.filter_cells(adata, min_genes=500)

    plots.plot_qc_distributions(adata)
    plt.show()

    print(
        f"Number of cells after filtering of low quality cells: {adata.n_obs}\n\n\n\n\n"
    )

    filtered_adata_list.append(adata)

In [ ]:
%%time
for adata in filtered_adata_list:
    gsm = adata.obs.GSM_id[0]
    if gsm in ("GSM5133599", "GSM5133600", "GSM5133604"):
        thr = 0.5
    elif gsm in (
        "GSM4058902",
        "GSM4058907",
        "GSM4058909",
        "GSM4058910",
        "GSM4058914",
        "GSM4058915",
        "GSM4058916",
        "GSM4058917",
        "GSM4058918",
        "GSM4143262",
        "GSM4143263",
        "GSM5388411",
        "GSM5388412",
        "GSM5388413",
        "GSM6509492",
        "GSM6509495",
        "GSM6598824",
        "GSM7103271",
        "GSM7103273",
        "GSM7103275",
        "GSM7103277",
        "GSM7103278",
        "GSM7103283",
        "GSM7103295",
        "GSM7103307",
        "GSM7135569",
        "GSM7135571",
        "GSM7135573",
    ):
        thr = 0.3
    elif gsm in (
        "GSM6598818",
        "GSM7103257",
        "GSM7103264",
        "GSM7135585",
        "GSM7135596",
        "GSM7135599",
        "GSM7135600",
        "GSM7135601",
    ):
        thr = 0.24
    else:
        thr = 0.4
    sce.pp.scrublet(
        adata, verbose=False, threshold=thr
    )  # batch_key="GSM_id" - is absent in this version
    sce.pl.scrublet_score_distribution(adata)
    print(gsm)
    plt.show()

In [ ]:
adata = ad.concat(filtered_adata_list, join="outer")

In [ ]:
adata.obs.sample(3)

In [ ]:
# Strip the numeric index suffixes and assign the GSM ID as the new suffix

adata_barcodes = adata.obs.index.str.extract(r"^([^-]+-\d+)")[0]
adata.obs["new_barcode"] = adata_barcodes.values + "-" + adata.obs["GSM_id"].astype(str)

adata.obs.set_index("new_barcode", inplace=True)
adata.obs_names.name = None

In [ ]:
adata.obs.sample(3)

In [ ]:
min_cells = int(0.0001 * adata.n_obs)  # 0.01% from total cells
sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
adata.write("./h_lung/sc/lung_QC_dbl_SenCIDenv.h5ad")

### Intestine

In [ ]:
samplesheet = pd.read_csv("./h_intestine/intestine_samplesheet.csv")
samplesheet.head(5)

In [ ]:
adata_list = []

In [ ]:
intestine_exclusion = ["GSM6090499"]  # by CR metrics

In [ ]:
for index, row in tqdm(samplesheet.iterrows(), total=len(samplesheet)):
    if row["GSM_id"] in intestine_exclusion:
        continue
    path = (
        f"./h_intestine/cellranger_out/{row['GSM_id']}/outs/filtered_feature_bc_matrix"
    )
    adata = sc.read_10x_mtx(path, prefix="")
    for col in row.index:
        adata.obs[col] = row[col]
    adata_list.append(adata)

In [ ]:
filtered_adata_list = []

In [ ]:
for num, adata in enumerate(adata_list):
    adata.obs.index = [f"{index}-{num}" for index in adata.obs.index]
    print(f"{num} - {adata.obs.GSM_id.unique()[0]}:")

    print(f"Total number of cells: {adata.n_obs}")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=None, log1p=True
    )

    plots.plot_qc_distributions(adata)
    plt.show()
    sc.pp.filter_cells(adata, min_counts=500)

    adata.obs["outlier"] = is_outlier(adata, "log1p_total_counts", 5) | is_outlier(
        adata, "n_genes_by_counts", 6
    )
    print(
        f"outlier counts: \n{adata.obs.outlier.value_counts().to_string(name=False, dtype=False)}\n"
    )

    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 4, "right") | (
        adata.obs["pct_counts_mt"] > 40
    )
    print(
        f"mt outlier counts: \n{adata.obs.mt_outlier.value_counts().to_string(name=False, dtype=False)}"
    )

    # filtering out:
    adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
    sc.pp.filter_cells(adata, min_genes=500)
    plots.plot_qc_distributions(adata)
    plt.show()

    print(
        f"Number of cells after filtering of low quality cells: {adata.n_obs}\n\n\n\n\n"
    )

    filtered_adata_list.append(adata)

In [ ]:
%%time
for adata in filtered_adata_list:
    gsm = adata.obs.GSM_id[0]
    sce.pp.scrublet(adata, verbose=False, threshold=0.21)
    sce.pl.scrublet_score_distribution(adata)
    print(gsm)
    plt.show()

In [ ]:
adata = ad.concat(filtered_adata_list, join="outer")

In [ ]:
adata.obs.sample(3)

In [ ]:
# Strip the numeric index suffixes and assign the GSM ID as the new suffix


adata_barcodes = adata.obs.index.str.extract(r"^([^-]+-\d+)")[0]
adata.obs["new_barcode"] = adata_barcodes.values + "-" + adata.obs["GSM_id"].astype(str)

adata.obs.set_index("new_barcode", inplace=True)
adata.obs_names.name = None

In [ ]:
min_cells = int(0.0001 * adata.n_obs)
sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
adata.write("./h_intestine/sc/intestine_QC_dbl_SenCIDenv.h5ad")

# Skin

In [ ]:
samplesheet = pd.read_csv("./h_skin/skin_samplesheet.csv")
samplesheet.head(5)

In [ ]:
adata_list = []

In [ ]:
for index, row in tqdm(samplesheet.iterrows(), total=len(samplesheet)):
    path = f"./h_skin/cellranger_out/{row['GSM_id']}/outs/filtered_feature_bc_matrix"
    adata = sc.read_10x_mtx(path, prefix="")
    for col in row.index:
        adata.obs[col] = row[col]
    adata_list.append(adata)

In [ ]:
filtered_adata_list = []

In [ ]:
for num, adata in enumerate(adata_list):
    adata.obs.index = [f"{index}-{num}" for index in adata.obs.index]
    print(f"{num} - {adata.obs.GSM_id.unique()[0]}:")

    print(f"Total number of cells: {adata.n_obs}")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=None, log1p=True
    )

    plots.plot_qc_distributions(adata)
    plt.show()

    sc.pp.filter_cells(adata, min_counts=500)

    adata.obs["outlier"] = is_outlier(adata, "log1p_total_counts", 5) | is_outlier(
        adata, "n_genes_by_counts", 6
    )
    print(
        f"outlier counts: \n{adata.obs.outlier.value_counts().to_string(name=False, dtype=False)}\n"
    )

    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 4, "right") | (
        adata.obs["pct_counts_mt"] > 40
    )
    print(
        f"mt outlier counts: \n{adata.obs.mt_outlier.value_counts().to_string(name=False, dtype=False)}"
    )

    # filtering out:
    adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

    sc.pp.filter_cells(adata, min_genes=500)

    plots.plot_qc_distributions(adata)
    plt.show()

    print(
        f"Number of cells after filtering of low quality cells: {adata.n_obs}\n\n\n\n\n"
    )

    filtered_adata_list.append(adata)

In [ ]:
%%time
for adata in filtered_adata_list:
    gsm = adata.obs.GSM_id[0]
    sce.pp.scrublet(adata, verbose=False, threshold=0.21)
    sce.pl.scrublet_score_distribution(adata)
    print(gsm)
    plt.show()

In [ ]:
adata = ad.concat(filtered_adata_list, join="outer")

In [ ]:
# Strip the numeric index suffixes and assign the GSM ID as the new suffix
adata_barcodes = adata.obs.index.str.extract(r"^([^-]+-\d+)")[0]
adata.obs["new_barcode"] = adata_barcodes.values + "-" + adata.obs["GSM_id"].astype(str)

adata.obs.set_index("new_barcode", inplace=True)
adata.obs_names.name = None

In [ ]:
min_cells = int(0.0001 * adata.n_obs)

sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
adata.write("./h_skin/sc/skin_QC_dbl_SenCIDenv.h5ad")

# Kidney (GSE211785 h5ad obj)

In [ ]:
obs = pd.read_csv("./h_kidney/GSE211785_export/obs.csv.gz", index_col=0)
var = pd.read_csv("./h_kidney/GSE211785_export/var.csv.gz", index_col=0)
X = sparse.load_npz("./h_kidney/GSE211785_export/X.npz")

In [ ]:
print("obs.shape:", obs.shape)
print("X.shape:", X.shape)
print("var.shape:", var.shape)

In [ ]:
adata_k = ad.AnnData(X=X, obs=obs, var=var)

In [ ]:
print(
    "adata.X: min =",
    adata_k.X.min(),
    ", max =",
    adata_k.X.max(),
    ", mean ≈",
    adata_k.X.mean(),
)
if adata_k.raw:
    print(
        "adata.raw.X: min =",
        adata_k.raw.X.min(),
        ", max =",
        adata_k.raw.X.max(),
        ", mean ≈",
        adata_k.raw.X.mean(),
    )

for key in adata_k.layers:
    layer = adata_k.layers[key]
    print(f"{key}: min = {layer.min()}, max = {layer.max()}, mean ≈ {layer.mean()}")
print("\n")

In [ ]:
adata_k.var = adata_k.var.loc[:, ~adata_k.var.columns.str.startswith("vst.")]

In [ ]:
# prep .obs w/o noisy cols
meta = pd.read_csv(
    "./h_kidney/GSE211785_scRNA-seq_snRNA-seq_snATAC-seq_metadata.txt", index_col=0
)
meta = meta[(meta.tech == "SC_RNA") & (meta.group == "Control")]
meta

In [ ]:
missing_in_meta = adata_k.obs.index.difference(meta.index)
extra_in_meta = meta.index.difference(adata_k.obs.index)

print(f"Total cells in adata: {adata_k.n_obs}.")
print("Cells in adata missing from meta:", len(missing_in_meta))
print("Rows in meta missing from adata:", len(extra_in_meta))
if len(missing_in_meta):
    print("Examples of missing indices:", list(missing_in_meta)[:5])
if len(extra_in_meta):
    print("Examples of extra indices in meta:", list(extra_in_meta)[:5])


In [ ]:
if not meta.index.equals(adata_k.obs.index):  # True if idx and their order are equal
    raise ValueError("indexes don't match.")
adata_k.obs = meta.loc[adata_k.obs.index].copy()  # KeyError if idx dont exist

In [ ]:
filtered_adata_list = []

In [ ]:
for num, sample in enumerate(adata_k.obs.orig_ident.unique()):
    adata = adata_k[adata_k.obs.orig_ident == sample].copy()
    print(f"{num} - {sample}:")

    print(f"Total number of cells: {adata.n_obs}")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=None, log1p=True
    )

    plots.plot_qc_distributions(adata)
    plt.show()

    sc.pp.filter_cells(adata, min_counts=500)

    adata.obs["outlier"] = is_outlier(adata, "log1p_total_counts", 5) | is_outlier(
        adata, "n_genes_by_counts", 7
    )
    print(
        f"outlier counts: \n{adata.obs.outlier.value_counts().to_string(name=False, dtype=False)}\n"
    )

    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 5, "right") | (
        adata.obs["pct_counts_mt"] > 40
    )
    print(
        f"mt outlier counts: \n{adata.obs.mt_outlier.value_counts().to_string(name=False, dtype=False)}"
    )

    # filtering out:
    adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

    sc.pp.filter_cells(adata, min_genes=300)

    plots.plot_qc_distributions(adata)
    plt.show()

    print(
        f"Number of cells after filtering of low quality cells: {adata.n_obs}\n\n\n\n\n"
    )

    filtered_adata_list.append(adata)

In [ ]:
for adata in filtered_adata_list:
    gsm = adata.obs.orig_ident[0]
    sce.pp.scrublet(adata, verbose=False, threshold=0.3)
    sce.pl.scrublet_score_distribution(adata)
    print(gsm)
    plt.show()

In [ ]:
adata = ad.concat(filtered_adata_list, join="outer")

In [ ]:
adata = adata[adata.obs.orig_ident != "HK2899.SC"]

In [ ]:
min_cells = int(0.0001 * adata.n_obs)  # 0.01% от общего числа клеток
sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
adata.write("./h_kidney/GSE211785_QC_dbl_SenCIDenv.h5ad")

# Heart

In [ ]:
obs = pd.read_csv("h_heart/export for sencid/obs.csv.gz", index_col=0)
var = pd.read_csv("h_heart/export for sencid/var.csv.gz", index_col=0)
X = sparse.load_npz("h_heart/export for sencid/X.npz")

In [ ]:
print("obs.shape:", obs.shape)
print("X.shape:", X.shape)
print("var.shape:", var.shape)

In [ ]:
adata = ad.AnnData(X=X, obs=obs, var=var)

In [ ]:
adata.var["ensg_name"] = adata.var.index

adata.var.set_index(keys="gene_name-new", inplace=True)

adata.var_names_make_unique()

In [ ]:
adata_full = adata.copy()

In [ ]:
adata_sc = adata[adata.obs.modality == "scRNA"]
adata_sn = adata[adata.obs.modality == "snRNA"]
adata_mu = adata[adata.obs.modality == "Multiome-RNA"]

In [ ]:
filtered_adata_list = []

In [ ]:
for adata_ in adata_sc, adata_sn, adata_mu:
    for num, sample in enumerate(adata_.obs.donor.unique()):
        adata = adata_[adata_.obs.donor == sample].copy()
        print(f"{num} - {sample}:")

        print(f"Total number of cells: {adata.n_obs}")
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
        sc.pp.calculate_qc_metrics(
            adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=None, log1p=True
        )

        plots.plot_qc_distributions(adata)
        plt.show()

        sc.pp.filter_cells(adata, min_counts=500)
        sc.pp.filter_cells(adata, max_counts=15000)

        plots.plot_qc_distributions(adata)
        plt.show()

        print(
            f"Number of cells after filtering of low quality cells: {adata.n_obs}\n\n\n\n\n"
        )

        filtered_adata_list.append(adata)

In [ ]:
adata = ad.concat(filtered_adata_list, join="outer")

In [ ]:
min_cells = int(0.0001 * adata.n_obs)  # 0.01% от общего числа клеток
sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
adata_full.obs.scrublet_score.plot()

In [ ]:
adata.write("./h_heart/heart_QC_dbl_SenCIDenv.h5ad")